# 15 — Graph-rewiring falsification
Tạo topology giả có kiểm chứng, sau đó chạy RR/RW/WW/WR. Output notebook là bằng chứng thực thi; manifest là bằng chứng toàn vẹn dữ liệu.

In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'research/server').is_dir(): raise RuntimeError('Hãy mở Jupyter tại thư mục gốc của repo')
def stage(name): subprocess.run(['bash', 'research/server/run_next_experiments.sh', name], cwd=ROOT, check=True)

In [ ]:
stage('rewire_prepare')

In [ ]:
manifest = json.loads((ROOT / 'research/results/graph_rewire_prepare.json').read_text())
assert manifest['complete'] is True
gate_rows = []
for dataset_name, splits in manifest['datasets'].items():
    for split_name, values in splits.items():
        assert values['protected_row_mismatches'] == 0
        assert values['destination_marginal_difference_rows'] == 0
        gate_rows.append({'dataset': dataset_name, 'split': split_name, **values})
display(pd.DataFrame(gate_rows))

In [ ]:
stage('rewire_eval_rw')

In [ ]:
stage('rewire_train')

In [ ]:
stage('rewire_verify')

In [ ]:
stage('rewire_eval_wr')

In [ ]:
rw = pd.read_csv(ROOT / 'research/artifacts/rewire_eval_RW/runs.csv')
ww = pd.read_csv(ROOT / 'research/artifacts/rewired_full_runs/runs.csv')
wr = pd.read_csv(ROOT / 'research/artifacts/rewire_eval_WR/runs.csv')
assert len(rw) == 120 and len(ww) == 80 and len(wr) == 80
display(rw.groupby(['dataset', 'task', 'model']).test_macro_f1.agg(['mean', 'std']))